# SustAInable — 03: Model Training + Evaluation
## Neighborhood Heat Illness Risk Prediction
---
**Purpose:** Split data, apply SMOTE to handle class imbalance, train an XGBoost 
classifier, and evaluate at decision threshold = 0.30.

**Key design decisions:**
- **Metric priority: Recall** — a false negative (missing a high-risk ZCTA) costs lives
- **SMOTE** oversampling on training set only (never on test set)
- **Decision threshold: 0.30** — deliberately lower than 0.50 to reduce false negatives
- **Asymmetric cost framing:** false negatives >> false positives in public health response


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, roc_curve, precision_recall_curve)
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

try:
    from xgboost import XGBClassifier
    print("✅ XGBoost available.")
except ImportError:
    print("⚠️  XGBoost not installed. Run: pip install xgboost")

try:
    from imblearn.over_sampling import SMOTE
    print("✅ imbalanced-learn available.")
except ImportError:
    print("⚠️  imbalanced-learn not installed. Run: pip install imbalanced-learn")

np.random.seed(42)


## Step 1 — Reconstruct Feature Matrix

In [ ]:
# Reconstruct merged feature matrix (mirrors NB 02 output)
N = 1200
np.random.seed(42)

poverty_rate     = np.random.beta(2,6,N)*0.6
disability_prev  = np.random.beta(2,5,N)*0.45
elderly_pct      = np.random.beta(2,5,N)*0.4
ac_access_pct    = np.clip(np.random.beta(5,2,N),0.2,1.0)
unhoused_rate    = np.random.beta(1.5,10,N)*0.08
urban_density    = np.random.choice([0,1,2],N,p=[0.2,0.5,0.3])
heat_index_delta = np.random.normal(3.5,1.8,N)
tree_canopy_pct  = np.clip(np.random.beta(3,4,N),0.01,0.6)
pct_no_vehicle   = np.random.beta(2,6,N)*0.5
median_income_k  = np.random.normal(52,18,N).clip(15,120)
region           = np.random.choice(['Northeast','Southeast','Midwest','Southwest','West'],
                                    N,p=[0.22,0.20,0.22,0.18,0.18])

risk_score = (0.25*poverty_rate+0.20*disability_prev+0.15*elderly_pct+
              0.15*(1-ac_access_pct)+0.10*(heat_index_delta/10)+
              0.08*unhoused_rate*5+0.07*(1-tree_canopy_pct))
label_prob = np.clip(1/(1+np.exp(-8*(risk_score-0.28)))+np.random.normal(0,0.04,N),0,1)

df = pd.DataFrame({
    'zcta':[f'{10000+i:05d}' for i in range(N)],
    'urban_density':urban_density,'poverty_rate':poverty_rate,
    'disability_prevalence':disability_prev,'elderly_pct':elderly_pct,
    'ac_access_pct':ac_access_pct,'unhoused_rate':unhoused_rate,
    'heat_index_delta':heat_index_delta,'tree_canopy_pct':tree_canopy_pct,
    'pct_no_vehicle':pct_no_vehicle,'median_income_k':median_income_k,
    'vulnerability_index':(0.3*poverty_rate+0.25*disability_prev+0.2*elderly_pct+0.25*(1-ac_access_pct)),
    'cooling_deficit':(1-ac_access_pct)*heat_index_delta,
    'mobility_barrier':pct_no_vehicle*(1-ac_access_pct),
    'income_heat_risk':heat_index_delta/(median_income_k/50),
    'elevated_heat_illness':(label_prob>0.5).astype(int)
})
df = pd.get_dummies(df, columns=['region'] if 'region' not in df.columns else [], drop_first=False)
# Add region dummies manually since region was created but not encoded above
for r in ['Northeast','Southeast','Midwest','Southwest','West']:
    df[f'region_{r}'] = (np.random.choice(['Northeast','Southeast','Midwest','Southwest','West'],N,p=[0.22,0.20,0.22,0.18,0.18]) == r).astype(int)

FEATURE_COLS = [c for c in df.columns if c not in ['zcta','elevated_heat_illness']]
X = df[FEATURE_COLS]
y = df['elevated_heat_illness']

print(f"Feature matrix: {X.shape}")
print(f"Label rate: {y.mean():.1%} positive")


## Step 2 — Train/Test Split + SMOTE

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {X_train.shape} | Test: {X_test.shape}")
print(f"Train label rate: {y_train.mean():.1%} | Test label rate: {y_test.mean():.1%}")

# SMOTE on training set only
try:
    sm = SMOTE(random_state=42)
    X_train_sm, y_train_sm = sm.fit_resample(X_train, y_train)
    print(f"\nAfter SMOTE — Train: {X_train_sm.shape}")
    print(f"SMOTE label rate: {y_train_sm.mean():.1%} (balanced)")
except Exception as e:
    print(f"SMOTE unavailable ({e}), using original training set.")
    X_train_sm, y_train_sm = X_train, y_train


## Step 3 — Train XGBoost Classifier

In [ ]:
try:
    model = XGBClassifier(
        n_estimators=200,
        max_depth=5,
        learning_rate=0.08,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=1,
        use_label_encoder=False,
        eval_metric='logloss',
        random_state=42,
        verbosity=0
    )
    model.fit(X_train_sm, y_train_sm,
              eval_set=[(X_test, y_test)],
              verbose=False)
    print("✅ XGBoost model trained.")
    y_prob = model.predict_proba(X_test)[:,1]
    y_pred_05 = (y_prob >= 0.50).astype(int)
    y_pred_03 = (y_prob >= 0.30).astype(int)
    print(f"\nROC-AUC: {roc_auc_score(y_test, y_prob):.3f}")
except Exception as e:
    print(f"XGBoost unavailable ({e}). Using logistic regression fallback.")
    from sklearn.linear_model import LogisticRegression
    model = LogisticRegression(max_iter=500, random_state=42)
    model.fit(X_train_sm, y_train_sm)
    y_prob = model.predict_proba(X_test)[:,1]
    y_pred_05 = (y_prob >= 0.50).astype(int)
    y_pred_03 = (y_prob >= 0.30).astype(int)
    print(f"\nROC-AUC (LR fallback): {roc_auc_score(y_test, y_prob):.3f}")


## Step 4 — Evaluate at Threshold 0.30

In [ ]:
print("=== THRESHOLD 0.50 ===")
print(classification_report(y_test, y_pred_05, target_names=['No Risk','Elevated Risk']))

print("\n=== THRESHOLD 0.30 (PRIMARY — reduces false negatives) ===")
print(classification_report(y_test, y_pred_03, target_names=['No Risk','Elevated Risk']))


In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_prob)
auc = roc_auc_score(y_test, y_prob)

prec, rec, _ = precision_recall_curve(y_test, y_prob)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].plot(fpr, tpr, color='#E53935', linewidth=2, label=f'XGBoost (AUC={auc:.3f})')
axes[0].plot([0,1],[0,1],'k--',alpha=0.5,label='Random')
axes[0].set_xlabel('False Positive Rate'); axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve', fontsize=13, fontweight='bold')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(rec, prec, color='#1565C0', linewidth=2)
axes[1].axvline(x=0.70, color='orange', linestyle='--', alpha=0.8, label='Target recall ≥0.70')
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve', fontsize=13, fontweight='bold')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('/tmp/03_model_curves.png', dpi=100, bbox_inches='tight')
plt.show()
print(f"ROC-AUC: {auc:.3f}")


In [ ]:
cm = confusion_matrix(y_test, y_pred_03)
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap='Blues')
plt.colorbar(im)
ax.set_xticks([0,1]); ax.set_yticks([0,1])
ax.set_xticklabels(['Pred: No Risk','Pred: Elevated Risk'])
ax.set_yticklabels(['True: No Risk','True: Elevated Risk'])
for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm[i,j]), ha='center', va='center',
                fontsize=18, fontweight='bold',
                color='white' if cm[i,j] > cm.max()/2 else 'black')
ax.set_title('Confusion Matrix (Threshold = 0.30)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('/tmp/03_confusion_matrix.png', dpi=100, bbox_inches='tight')
plt.show()


---
## Notebook Summary
- XGBoost trained with SMOTE-balanced training data
- Decision threshold = 0.30 to prioritize recall (minimize missed high-risk ZCTAs)
- **Next:** `04_shap_explainability.ipynb` — SHAP values for model interpretability
